In [29]:
using LinearAlgebra

In [30]:
"""
    calcular_coefs_chebyshev(f, limit_k)

Calcula numéricamente los coeficientes de la serie de Chebyshev \$a_k\$ para una función \$f(x)\$ definida en el intervalo [-1, 1].

Esta función aproxima la integral definida:

\$\$a_k = \\frac{2}{\\pi} \\int_{0}^{\\pi} f(\\cos(\\theta)) \\cos(k\\theta) \\, d\\theta\$\$

Utiliza la **Regla del Trapecio compuesta** con 1000 nodos para realizar la integración numérica

# Argumentos
- `f`: La función matemática objetivo (ej. `x -> exp(x)`).
- `limit_k`: El índice máximo hasta el cual calcular (se devolverán `limit_k + 1` coeficientes, desde \$a_0\$ hasta \$a_{limit\\_k}\$).

# Retorna
- `Vector{Float64}`: Un vector con los coeficientes calculados `[a_0, a_1, ..., a_limit_k]`.
"""
function calcular_coefs_chebyshev(f, limit_k)
    coeffs = Float64[]
    n_nodos = 1000
    thetas = range(0, π, length=n_nodos+1)
    dt = π / n_nodos
    
    for k in 0:limit_k
        integral = 0.0
        for θ in thetas
            val = f(cos(θ)) * cos(k * θ)
            w = (θ == 0 || θ == π) ? 0.5 : 1.0
            integral += val * w
        end
        ak = (2/π) * integral * dt
        push!(coeffs, ak)
    end
    return coeffs
end

calcular_coefs_chebyshev

In [31]:
"""
    chebyshev_racional(f::Function, n::Int, m::Int)

Calcula los coeficientes de los polinomios \$P(x)\$ y \$Q(x)\$ para la **Aproximación Racional de Chebyshev** de una función \$f(x)\$.

La aproximación resultante tiene la forma:
\$\$r_T(x) = \\frac{\\sum_{k=0}^n p_k T_k(x)}{\\sum_{k=0}^m q_k T_k(x)}\$\$

# Descripción del Método
Implementa el algoritmo:
1.  **Integración:** Calcula los coeficientes de Chebyshev \$a_k\$ de \$f(x)\$ mediante integración numérica.
2.  **Sistema Lineal (Denominador):** Construye y resuelve un sistema lineal para encontrar los coeficientes \$q_k\$ que anulan los términos de orden superior (\$T_{n+1}\$ a \$T_{n+m}\$).
3.  **Numerador:** Calcula los coeficientes \$p_k\$ mediante una suma ponderada de los \$a_k\$ y \$q_k\$.
4.  **Corrección:** Ajusta el término \$p_0\$ dividiéndolo por 2 (necesario por la definición del producto de polinomios \$T_0\cdot T_k\$).

# Argumentos
- `f`: La función matemática a aproximar (ej. `x -> exp(x)`).
- `n`: Grado del polinomio del numerador \$P(x)\$.
- `m`: Grado del polinomio del denominador \$Q(x)\$.

# Retorna
- `(p, q)`: Una tupla donde `p` y `q` son vectores que contienen los coeficientes calculados.
"""
function chebyshev_racional(f::Function, n::Int, m::Int)
    #Determinar N
    N = n + m
    
    #Coeficientes a_k
    coeffs_a = calcular_coefs_chebyshev(f, N + m)
    
    # Función para acceder a a_k
    function get_a(k)
        idx = abs(k)
        return coeffs_a[idx + 1]
    end

    #Determinar q_0 = 1 
    #Determinar los coeficientes q_1 ... q_m 
    if m > 0
        A = zeros(m, m)
        b = zeros(m)
        
        for i in 1:m
            #Ecuaciones para los índices k = n + 1, ..., n + m
            k = n + i 
            
            for j in 1:m
                # Determinar coeficientes de la matriz
                val = 0.5 * (get_a(k + j) + get_a(k - j))
                A[i, j] = val
            end
            
            #El lado derecho mueve el término q_0, que es 1
            b[i] = -get_a(k)
        end
        
        #Resolución del sistema
        try
            q_values = A \ b
            q = [1.0; q_values] # Concatenamos q_0 = 1
        catch
            error("El sistema es singular")
        end
    else
        q = [1.0]
    end

    #Calcular p_0 ... p_n
    p = zeros(n + 1)
    
    for i in 0:n
        #Calculamos el coeficiente i-esimo del numerador
        sum_val = 0.0
        for j in 0:m
            # Sumamos q_j * 0.5 * (a_{i+j} + a_{|i-j|})
            term = 0.5 * (get_a(i + j) + get_a(i - j))
            sum_val += q[j+1] * term
        end
        p[i+1] = sum_val
    end
    p[1] = p[1] / 2.0

    return p, q
end


Base.Meta.ParseError: ParseError:
# Error @ /Users/mikitl/Documents/FES/Ciencia de Datos/Clases/Metodos_Matematicos/Julia/DATASCIENCE/Quinto/Metodos Numericos @ CD/Parte 2/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W1sZmlsZQ==.jl:14:127
3.  **Numerador:** Calcula los coeficientes \$p_k\$ mediante una suma ponderada de los \$a_k\$ y \$q_k\$.
4.  **Corrección:** Ajusta el término \$p_0\$ dividiéndolo por 2 (necesario por la definición del producto de polinomios \$T_0\cdot T_k\$).
#                                                                                                                             └┘ ── invalid escape sequence

In [32]:
f(x) = exp(x) #Función a comprobar


p, q = chebyshev_racional(f, 3, 3)

println("Numerador P: ", p)
println("Denominador Q: ", q)


# Aproximación
aprox = sum(p) / sum(q)
real = exp(1.0)

println("\nX = 1")
println("Aproximación: $aprox")
println("Valor Real:   $real")
println("Error:        $(abs(aprox - real))")

Numerador P: [1.6955467538271938, 2.128441192425524, 0.4191515662365096, 0.03282691990423368]
Denominador Q: [1.0, 0.8254113054328015, -0.27637385976808065, 0.02401657575810471]

X = 1
Aproximación: 2.718257843761687
Valor Real:   2.718281828459045
Error:        2.3984697357981588e-5
